# 02 — Train one (arm, fold)

**One notebook run = one of the 36 runs.** Set `ARM`, `FOLD` and `NEGATIVE_CONTROL` in the
parameters cell, then *Save & Run All*. Kaggle keeps the outputs, so a session dying costs
one run rather than the sweep.

Nothing is tunable here. Every hyperparameter comes from `configs/model.yaml` in the pinned
commit and every decode setting from the frozen `DecodeConfig`. `ARM` selects a data file
and nothing else — that is what makes the six arms comparable.

**Attach:** dataset `emocap-v2-arms`; **Accelerator:** GPU; **Internet:** on (to clone the
pinned commit).

The 36 runs are: 6 arms × 5 folds, plus one negative control per arm on fold 0.

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
ARM = "S_unpaired"       # S_paired25 | S_paired5 | S_unpaired | V1_paired5 | V1_unpaired | H_unpaired
FOLD = 0                  # 0..4
NEGATIVE_CONTROL = False  # True runs the registered shuffled-label control (fold 0 only)
SEED = 42

DATA = "/kaggle/input/emocap-v2-arms"
OUT = "/kaggle/working/runs"

In [ ]:
# Clone the EXACT commit the data was built from. Pinning to the commit recorded in
# provenance.json is what stops a notebook from pairing this dataset version with a
# different version of the code -- a mismatch that would be silent and unrecoverable.
import json, os, subprocess
from pathlib import Path

COMMIT = json.loads(Path(DATA, "provenance.json").read_text())["git_commit"]
if not Path("/kaggle/working/EmoCap").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/asjad2401/EmoCap.git",
                    "/kaggle/working/EmoCap"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/EmoCap", "checkout", "-q", COMMIT], check=True)
print("code at", COMMIT[:12])

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/EmoCap/src")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# The training script is the same one that runs locally -- no notebook-only code path,
# so a result can be reproduced off Kaggle.
cmd = [sys.executable, "/kaggle/working/EmoCap/scripts/train_arm.py",
       "--arm", ARM, "--fold", str(FOLD), "--seed", str(SEED),
       "--data-root", DATA, "--out-root", OUT]
if NEGATIVE_CONTROL:
    cmd.append("--negative-control")
print(" ".join(cmd), "\n")
!{" ".join(cmd)}

In [ ]:
# What landed. `empty_captions` above zero means decode collapsed and the run is suspect.
tag = f"{ARM}-f{FOLD}" + ("-nc" if NEGATIVE_CONTROL else "")
run = Path(OUT, tag)
print(json.dumps(json.loads((run / "stats.json").read_text()), indent=2))

preds = [json.loads(l) for l in (run / "predictions.jsonl").read_text().splitlines()]
print(f"\n{len(preds):,} predictions. First few per register:")
seen = set()
for p in preds:
    if p["emotion"] not in seen:
        seen.add(p["emotion"])
        print(f"  [{p['emotion']:<9}] {p['generated']}")
        print(f"  {'':>11} ref: {p['reference']}")